In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
df_samples = pd.read_csv("/home/earkfeld/Projects/MitoSpace4D/runs/embeddings_test/image_paths.csv", dtype=str, header=None, delimiter=" ")
df_regions = pd.read_csv("/run/user/1002/gvfs/smb-share:server=aquila0.jslab.ucsd.edu,share=ssd_processing/Others/MitoSpace4D/2025_summer/cell_to_region.csv")

In [3]:
# Set the header for the first column
df_samples.columns = ["fpath"]
df_samples['condition'] = df_samples['fpath'].apply(lambda x: x.split("/")[-2])
df_samples['filename'] = df_samples['fpath'].apply(lambda x: x.split("/")[-1].split(".")[0])
df_samples['cell_id'] = df_samples['filename'].apply(lambda x: int(x.split("-")[0]))
df_samples['cell_tid'] = df_samples['filename'].apply(lambda x: int(x.split("-")[1]))

# Set up an empty column called "condition_tid"
df_samples['region_id'] = -1

In [4]:
df_samples

,fpath,condition,filename,cell_id,cell_tid,region_id
0,/run/user/1002/gvfs/smb-share:server=aquila0.j...,20250724-1,000169-0,169,0,-1
1,/run/user/1002/gvfs/smb-share:server=aquila0.j...,20250724-1,000501-1,501,1,-1
2,/run/user/1002/gvfs/smb-share:server=aquila0.j...,20250724-1,000430-1,430,1,-1
3,/run/user/1002/gvfs/smb-share:server=aquila0.j...,20250724-1,000167-0,167,0,-1
4,/run/user/1002/gvfs/smb-share:server=aquila0.j...,20250724-1,000310-0,310,0,-1
...,...,...,...,...,...,...
1546,/run/user/1002/gvfs/smb-share:server=aquila0.j...,20250724-1,000287-1,287,1,-1
1547,/run/user/1002/gvfs/smb-share:server=aquila0.j...,20250724-1,000271-0,271,0,-1
1548,/run/user/1002/gvfs/smb-share:server=aquila0.j...,20250724-1,000287-0,287,0,-1
1549,/run/user/1002/gvfs/smb-share:server=aquila0.j...,20250724-1,000350-1,350,1,-1


In [5]:
for condition in df_regions["Data Path"].unique():
    df_regions_condition = df_regions[df_regions["Data Path"] == condition]
    df_samples_condition = df_samples[df_samples["condition"] == condition]

    for i, row in df_regions_condition.iterrows():
        cell_id_start = row["Cell ID Start"]
        try:
            cell_id_end = df_regions_condition.loc[i + 1, "Cell ID Start"]
        except:
            # Set infinite integer value
            cell_id_end = np.inf

        current_region_id = row["Region ID"]
        for i, sample_row in df_samples_condition.iterrows():
            current_cell_id = sample_row['cell_id']

            if current_cell_id >= cell_id_start and current_cell_id < cell_id_end:
                df_samples_condition.at[i, "region_id"] = current_region_id

    df_samples.update(df_samples_condition)

In [6]:
time_id_fn = lambda region_id, cell_tid: 3*region_id + cell_tid
df_samples["time_id"] = df_samples.apply(lambda x: time_id_fn(x["region_id"], x["cell_tid"]), axis=1)

In [8]:
df_samples_sorted = df_samples.sort_values(by=["condition", "filename", "time_id"], inplace=False)

In [10]:
df_samples_sorted

,fpath,condition,filename,cell_id,cell_tid,region_id,time_id
217,/run/user/1002/gvfs/smb-share:server=aquila0.j...,20250722-2,000000-0,0,0,0,0
4266,/run/user/1002/gvfs/smb-share:server=aquila0.j...,20250722-2,000000-1,0,1,0,1
883,/run/user/1002/gvfs/smb-share:server=aquila0.j...,20250722-2,000000-2,0,2,0,2
2573,/run/user/1002/gvfs/smb-share:server=aquila0.j...,20250722-2,000001-0,1,0,0,0
7316,/run/user/1002/gvfs/smb-share:server=aquila0.j...,20250722-2,000001-1,1,1,0,1
...,...,...,...,...,...,...,...
6601,/run/user/1002/gvfs/smb-share:server=aquila0.j...,20250806-2,000352-1,352,1,7,22
10969,/run/user/1002/gvfs/smb-share:server=aquila0.j...,20250806-2,000352-2,352,2,7,23
9146,/run/user/1002/gvfs/smb-share:server=aquila0.j...,20250806-2,000353-0,353,0,7,21
2754,/run/user/1002/gvfs/smb-share:server=aquila0.j...,20250806-2,000353-1,353,1,7,22


In [13]:
# Set up a plasma color map as a function of time_id (rgb, 0-255)
df_samples['color'] = df_samples['time_id'].apply(lambda x: (np.array(plt.cm.plasma(x / df_samples['time_id'].max())[:3]) * 255).astype(int))

In [22]:
colors_arr = np.vstack(df_samples['color'].values)
np.save('temporal_colormap.npy', colors_arr)

0
